# Desafio RAG — 3. Rerank RAG

A busca vetorial é rápida, mas compara pergunta e trecho **separadamente** (bi-encoder), o que às vezes traz trechos só superficialmente parecidos.

O Rerank RAG faz em duas etapas: primeiro uma busca vetorial **ampla** (ex.: 25 candidatos) e depois um **cross-encoder** lê pergunta + trecho **juntos** e dá uma nota de relevância mais precisa. Só os melhores vão ao LLM.

```
pergunta → top-25 vetorial → reranker (cross-encoder) → top-5 → LLM → resposta
```

## 1. Instalação

In [ ]:
# Instala as dependências (no Colab leva ~1 min)
!pip install -q pypdf chromadb sentence-transformers langchain-text-splitters google-genai

## 2. Configuração

Crie uma chave gratuita em [Google AI Studio](https://aistudio.google.com/apikey). No Colab, adicione-a em **Secrets** (ícone de chave à esquerda) com o nome `GEMINI_API_KEY`. Use um ambiente com **GPU** (Ambiente de execução → Alterar tipo) para os embeddings rodarem rápido.

In [ ]:
import os, re, json, time, bisect, getpass
import requests
import numpy as np

# ---- LLM (Google Gemini — tem camada gratuita: https://aistudio.google.com/apikey)
LLM_MODEL = "gemini-2.5-flash"   # troque se quiser outro modelo Gemini

try:  # no Colab: guarde a chave em "Secrets" com o nome GEMINI_API_KEY
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("GEMINI_API_KEY: ")

# ---- Embeddings multilíngues (bom para português)
EMBED_MODEL = "intfloat/multilingual-e5-base"

PDF_URL  = "https://fundar.org.br/wp-content/uploads/2021/06/os-sertoes.pdf"
PDF_PATH = "os-sertoes.pdf"

QUESTIONS = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

## 3. Carregamento e limpeza do PDF

In [ ]:
from pypdf import PdfReader

if not os.path.exists(PDF_PATH):
    r = requests.get(PDF_URL, timeout=120, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(r.content)
# Se o download falhar, baixe o PDF manualmente e faça upload com o nome os-sertoes.pdf

def clean(text: str) -> str:
    text = re.sub(r"-\n(\w)", r"\1", text)        # junta palavras hifenizadas na quebra de linha
    text = re.sub(r"(?<![.!?:;])\n(?!\n)", " ", text)  # quebra de linha no meio de frase vira espaço
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

reader = PdfReader(PDF_PATH)
pages = [(i + 1, clean(p.extract_text() or "")) for i, p in enumerate(reader.pages)]
pages = [(n, t) for n, t in pages if len(t) > 50]   # descarta páginas vazias/capa

# Texto contínuo + mapa de offsets -> página (para citar a fonte de cada trecho)
full_text, page_starts, page_nums = "", [], []
for n, t in pages:
    page_starts.append(len(full_text)); page_nums.append(n)
    full_text += t + "\n\n"

def page_of(offset: int) -> int:
    return page_nums[bisect.bisect_right(page_starts, offset) - 1]

print(f"{len(reader.pages)} páginas no PDF, {len(pages)} com texto, {len(full_text):,} caracteres")

## 4. Embeddings e banco vetorial

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split(text, size, overlap, base_offset=0):
    """Divide o texto e devolve [(trecho, offset_inicial)]."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""],
        add_start_index=True,
    )
    docs = splitter.create_documents([text])
    return [(d.page_content, base_offset + d.metadata["start_index"]) for d in docs]

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)

# O modelo e5 espera os prefixos "passage: " e "query: "
def embed_passages(texts):
    return embedder.encode([f"passage: {t}" for t in texts], batch_size=32,
                           normalize_embeddings=True, show_progress_bar=True).tolist()

def embed_query(q):
    return embedder.encode(f"query: {q}", normalize_embeddings=True).tolist()

chroma = chromadb.Client()   # em memória

def build_collection(name, texts, metadatas):
    try: chroma.delete_collection(name)
    except Exception: pass
    col = chroma.create_collection(name, metadata={"hnsw:space": "cosine"})
    embs = embed_passages(texts)
    for i in range(0, len(texts), 1000):   # insere em lotes
        col.add(ids=[str(j) for j in range(i, min(i + 1000, len(texts)))],
                documents=texts[i:i+1000], embeddings=embs[i:i+1000], metadatas=metadatas[i:i+1000])
    return col

## 5. Indexação, recuperação e reranking

Mesmos trechos do Naive (1000/200), mais o reranker multilíngue `BAAI/bge-reranker-v2-m3`.

In [ ]:
RAG_NAME = "rerank"
CHUNK_SIZE, CHUNK_OVERLAP = 1000, 200
FETCH_K, TOP_K = 25, 5
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

chunks = split(full_text, CHUNK_SIZE, CHUNK_OVERLAP)
texts = [t for t, _ in chunks]
metas = [{"page": page_of(off)} for _, off in chunks]
collection = build_collection("rerank", texts, metas)
print(f"{len(texts)} trechos indexados")

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANK_MODEL, max_length=512)

def retrieve(question, fetch_k=FETCH_K, k=TOP_K, verbose=False):
    res = collection.query(query_embeddings=[embed_query(question)], n_results=fetch_k)
    cands = [{"text": d, "page": m["page"], "vec_rank": i + 1}
             for i, (d, m) in enumerate(zip(res["documents"][0], res["metadatas"][0]))]
    scores = reranker.predict([(question, c["text"]) for c in cands])
    for c, s in zip(cands, scores):
        c["score"] = float(s)
    top = sorted(cands, key=lambda c: c["score"], reverse=True)[:k]
    if verbose:
        for c in top:
            print(f"  score {c['score']:.3f} | posição na busca vetorial: {c['vec_rank']:>2} | p. {c['page']}")
    return top

# Veja como o reranker reordena os candidatos
_ = retrieve(QUESTIONS[0], verbose=True)

## 6. Geração com o LLM

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

SYSTEM = (
    "Você é um especialista em literatura brasileira e responde perguntas sobre o livro "
    "'Os Sertões', de Euclides da Cunha. Use APENAS os trechos fornecidos no contexto. "
    "Responda em português, de forma organizada e fundamentada, citando as páginas entre "
    "colchetes, ex.: [p. 42]. Se o contexto não for suficiente, diga claramente o que falta."
)

def generate(question, contexts):
    ctx = "\n\n".join(f"[p. {c['page']}]\n{c['text']}" for c in contexts)
    prompt = f"CONTEXTO:\n{ctx}\n\nPERGUNTA: {question}\n\nRESPOSTA:"
    for attempt in range(5):
        try:
            resp = client.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.2),
            )
            return resp.text
        except Exception as e:   # limite de requisições da camada gratuita
            wait = 15 * (attempt + 1)
            print(f"  erro na API ({e.__class__.__name__}), tentando de novo em {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Falha ao chamar o LLM")

## 7. Respondendo às 5 questões

In [ ]:
from IPython.display import Markdown, display

def rag(question):
    contexts = retrieve(question)
    return generate(question, contexts), contexts

results = []
for i, q in enumerate(QUESTIONS, 1):
    answer, ctxs = rag(q)
    pages_used = sorted({c["page"] for c in ctxs})
    results.append({"pergunta": q, "resposta": answer, "paginas": pages_used})
    display(Markdown(f"### {i}. {q}\n\n{answer}\n\n*Páginas recuperadas: {pages_used}*\n\n---"))

## 8. Salvando as respostas

O JSON gerado permite comparar lado a lado os três notebooks.

In [ ]:
with open(f"respostas_{RAG_NAME}.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Salvo em respostas_{RAG_NAME}.json")

## 9. Inspeção dos trechos recuperados

In [ ]:
# Inspecione o que foi recuperado para uma pergunta (útil para comparar as abordagens)
for c in retrieve(QUESTIONS[3])[:3]:
    print(f"--- p. {c['page']} ---\n{c['text'][:600]}...\n")